# Neon 기업별 데이터 보유 현황
두 Agent에 필요한 데이터의 기간·건수·누락·추가 확보 자료를 읽기 전용으로 점검합니다.

**실제 DB 결과는 실행 후 확인됩니다. 이 노트북을 만든 시점에는 DB에 접속하지 않았습니다.**

프로젝트 또는 notebooks 폴더에서 실행하세요. `agent_reader` 접속 주소를 로컬 `.env`의 DATABASE_URL에 설정합니다.
주소·비밀번호를 코드 셀에 붙여 넣거나 출력하지 마세요. 공용 배포 전 노트북 출력을 지우세요.


In [ ]:
# 최초 한 번, 필요한 경우 주석 해제
# %pip install "psycopg[binary]>=3.2,<4" "python-dotenv>=1,<2" "pandas>=2,<3"


In [ ]:
import os
import sys
from datetime import date
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "sh_agents/data_inventory.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("압축 해제한 프로젝트 폴더에서 노트북을 실행하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT.parents[1] / ".env", override=False)
from sh_agents.data_inventory import Inventory, acquisition_plan, open_connection

print("DATABASE_URL 설정:", "있음" if os.getenv("DATABASE_URL") else "없음")


## 1. 조사 범위
종목코드는 문자열로 지정합니다. 기본은 삼성전자·SK하이닉스이며 비교기업을 더할 수 있습니다(최대 20개).
날짜는 데이터의 관측 기간입니다. 과거 당시 공개됐던 자료만을 복원하는 백테스트가 아닙니다.
공시 기준은 접수일, 문서 기준은 공개일, 재무 기준은 period_end입니다. 기간 기준을 서로 혼동하지 마세요.


In [ ]:
TICKERS = ["005930", "000660"]
MARKET = "KR"
START = "2024-10-27"
END = "2026-10-27"  # 분석할 날짜로 수정
EXPECTED_ROLE = "agent_reader"  # 팀의 다른 읽기 전용 역할을 쓰면 명시적으로 수정
assert date.fromisoformat(START) <= date.fromisoformat(END)


## 2. 실제 DB 점검
스키마를 조회하여 존재하는 필드만 검사합니다. 보이지 않는 테이블은 '스키마/권한 확인 필요'로 표시합니다.
연결 시 읽기 전용 세션·15초 쿼리 제한·10초 연결 제한을 적용합니다. 각 셀 끝에 연결이 닫힙니다.


In [ ]:
with open_connection(EXPECTED_ROLE) as conn:
    inventory = Inventory(conn)
    companies = inventory.companies(TICKERS, MARKET)
    coverage_rows = inventory.coverage(companies, START, END)

found = {c["ticker"] for c in companies}
company_df = pd.DataFrame(companies)
display(company_df)
print("미조회 종목:", sorted(set(TICKERS) - found) or "없음")
coverage_df = pd.DataFrame(coverage_rows)
if coverage_df.empty:
    print("조회할 기업 없음. 종목코드·시장·마스터 적재를 확인하세요.")
else:
    display(coverage_df)


## 기업별 두 Agent 실행 준비 판정
집계만으로 원문 내용·가격 조정을 보증할 수 없어 정상 집계도 일부 가능으로 표시합니다. 핵심 미보유·조회 실패·오래된 시세·결측·최소 기간 부족은 보류합니다. 실제 실행 전 원시 자료 검사를 추가하세요.


In [ ]:
from sh_agents.quality_gate import inventory_gate_table

READINESS_REFERENCE_DATE = "2026-10-27"  # 실제 점검일로 수정
readiness = inventory_gate_table(coverage_rows, start=START, end=END, reference_date=READINESS_REFERENCE_DATE, window=3)
display(pd.DataFrame(readiness))


## 3. 누락·추가 수집 계획
NULL은 미확인 값이며 0과 다릅니다. '필드 결측 건수 합'은 필드별 NULL 합계로, 결측 행 수와 다를 수 있습니다.
날짜 결측은 기간에 배정할 수 없어 해당 기업 전체에서 집계합니다.
기간 내 보유는 완전 수집·검증 완료를 뜻하지 않습니다. 최신일이 요청 종료일보다 이전이어도 휴장·공개주기를 먼저 확인합니다.


In [ ]:
plan_df = pd.DataFrame(acquisition_plan(coverage_rows))
display(plan_df)
if not coverage_df.empty:
    issues = coverage_df[(coverage_df["상태"] != "기간 내 보유")
                         | coverage_df["추가 확보"].ne("")
                         | coverage_df["날짜 결측"].fillna(0).gt(0)
                         | coverage_df["필드 결측 건수 합"].fillna(0).gt(0)]
    display(issues)


## 4. 선택 사항 — 정확한 시세 누락 거래일
같은 시장·거래소·수집 범위의 공식 거래일 CSV를 준비합니다. 열 이름은 `trade_date`, 형식은 YYYY-MM-DD입니다.
주말을 제외한 날짜를 공식 거래일로 가정하지 않습니다. 상장 전·상장폐지 후는 회사별 범위를 조정하세요.
공식 달력이 없으면 아래 셀은 건너뛰며 정확한 누락 일수를 주장하지 않습니다.


In [ ]:
CALENDAR_CSV = None  # 예: ROOT / "private_data/trading_calendar.csv"
if CALENDAR_CSV is None:
    print("공식 거래일 달력 미설정: 정확한 누락일 판정 보류")
else:
    calendar = pd.read_csv(CALENDAR_CSV, dtype=str)
    days = pd.to_datetime(calendar["trade_date"], errors="raise").dt.date.astype(str).tolist()
    with open_connection(EXPECTED_ROLE) as conn:
        inventory = Inventory(conn)
        gaps = [inventory.price_gaps(c, START, END, days) for c in companies]
    display(pd.DataFrame(gaps))


## 5. 선택 사항 — 로컬 표 저장
DB에는 쓰지 않습니다. CSV에는 기업별 보유 현황이 담기므로 팀의 공유 범위를 확인하세요.


In [ ]:
EXPORT = False
if EXPORT:
    out = ROOT / "private_outputs/inventory"
    out.mkdir(parents=True, exist_ok=True)
    coverage_df.to_csv(out / "coverage.csv", index=False, encoding="utf-8-sig")
    plan_df.to_csv(out / "collection_plan.csv", index=False, encoding="utf-8-sig")
    print("로컬 CSV 저장 완료")


## 해석의 한계
- OHLCV의 NULL과 날짜 중복은 확인할 수 있지만 가격 조정·분할 반영 여부는 원천 문서로 확인해야 합니다.
- 재무표 보유는 제품·지역별 매출·경쟁 강도 자료 확보를 뜻하지 않습니다.
- rag.document의 건수·URL은 원문 위치·사용 권한·검색 품질의 검증이 아닙니다. 청크·임베딩 품질은 별도 점검입니다.
- period_end가 없는 재무 원계정 테이블은 조회를 건너뜁니다. fiscal_year를 임의 날짜로 변환하지 않습니다.
- 연결 오류가 나면 접속 주소를 출력하지 말고 계정·SSL·네트워크를 확인하세요.
